# Интерпретация: SHAP и причины отказа
Нужно: pip install shap joblib
Модель обучаем на dev с подобранными параметрами. Hold-out не трогаем.

Вклады SHAP считаем встроенным в LightGBM алгоритмом TreeSHAP (`pred_contrib=True`):
он даёт те же значения, что shap.TreeExplainer, но не зависит от версии shap.
Вклады в лог-шансах дефолта: положительный вклад увеличивает риск.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.special import expit

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.explain import EXCLUDED_GROUPS, GROUPS, build_mapping, group_shap, top_reasons
from src.features import ID_COL, TARGET_COL
from src.models import make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
N_SAMPLE = 20_000

## 1. Модель на dev

In [ ]:
df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
X = to_lgbm(dev.drop(columns=[TARGET_COL, ID_COL]))

cfg = json.loads((MODELS / "lgbm_best_params.json").read_text())
model = make_lgbm(
    n_estimators=cfg["n_estimators"], learning_rate=cfg["learning_rate"], **cfg["params"]
).fit(X, y)
joblib.dump(model, MODELS / "lgbm_dev.joblib")
print("Модель обучена, деревьев:", cfg["n_estimators"])

## 2. SHAP на выборке из dev

In [ ]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X), size=N_SAMPLE, replace=False)
Xs = X.iloc[sample_idx].reset_index(drop=True)
ys = y.iloc[sample_idx].reset_index(drop=True)

contrib = model.booster_.predict(Xs, pred_contrib=True)  # (n, признаки + 1)
sv, base = contrib[:, :-1], contrib[:, -1]
proba = model.predict_proba(Xs)[:, 1]

# проверка аддитивности: base + сумма вкладов = лог-шансы модели
print("Макс. расхождение с моделью:", np.abs(expit(base + sv.sum(axis=1)) - proba).max())

# для графиков категории переводим в числовые коды
Xs_plot = Xs.copy()
for c in Xs_plot.select_dtypes("category").columns:
    Xs_plot[c] = Xs_plot[c].cat.codes.astype(float)

## 3. Глобальная картина по признакам

In [ ]:
shap.summary_plot(sv, Xs_plot, max_display=20, show=False)
plt.title("SHAP: влияние признаков на риск дефолта")
plt.tight_layout()
plt.show()

In [ ]:
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=Xs.columns).sort_values(ascending=False)
share = (mean_abs / mean_abs.sum() * 100).round(2)
print("Топ-20 признаков по среднему |SHAP| (% от суммы):")
print(share.head(20))

## 4. Сопоставление признаков с причинами
Проверяем, насколько правила в src/explain.py покрывают важные признаки.

In [ ]:
mapping = build_mapping(Xs.columns)
by_group = share.groupby(mapping.reindex(share.index).values).sum().sort_values(ascending=False)
by_group.index = [f"{g}: {GROUPS[g]}" for g in by_group.index]
print("Вклад групп, % от суммы |SHAP| по признакам:")
print(by_group.round(1))

excluded_share = share[mapping.reindex(share.index).isin(EXCLUDED_GROUPS).values].sum()
print(f"\nДоля вклада признаков, которые мы не показываем клиенту: {excluded_share:.1f}%")

print("\nНе сопоставлены (группа 'other'), топ-15:")
print(share[mapping.reindex(share.index).values == "other"].head(15))

In [ ]:
# Важность групп (в сумме, а не по отдельным признакам): сколько вклада прячется в коррелированных признаках
fig, ax = plt.subplots(figsize=(9, 6))
by_group.sort_values().plot.barh(ax=ax)
ax.set(title="Вклад групп признаков (% суммы |SHAP|)", xlabel="%")
plt.tight_layout()
plt.show()

## 5. Причины отказа: топ-3 для каждого клиента
Вклады признаков суммируются по группам (SHAP аддитивен), группы из EXCLUDED_GROUPS пропускаются.

In [ ]:
grouped = group_shap(sv, Xs.columns)
reasons = top_reasons(grouped, k=3, min_shap=0.0)

# высокорисковый клиент, низкорисковый и "средний"
order = np.argsort(proba)
examples = {"высокий риск": order[-1], "средний риск": order[len(order) // 2], "низкий риск": order[0]}
for label, i in examples.items():
    print(f"--- {label}: вероятность дефолта {proba[i]:.1%} | факт: {ys.iloc[i]} ---")
    if reasons[i]:
        for text, val in reasons[i]:
            print(f"  {text}  (вклад {val:+.2f})")
    else:
        print("  значимых причин риска нет")
    print()

In [ ]:
# Waterfall для высокорискового клиента (по отдельным признакам)
i = examples["высокий риск"]
shap.plots.waterfall(
    shap.Explanation(values=sv[i], base_values=base[i], data=Xs_plot.iloc[i].values,
                     feature_names=list(Xs.columns)),
    max_display=12,
)

## 6. Проверка причин на здравый смысл

In [ ]:
# Какие причины чаще всего оказываются в топ-3 у самых рисковых 10% клиентов
risky = np.where(proba >= np.quantile(proba, 0.9))[0]
counts = pd.Series([t for i in risky for t, _ in reasons[i]]).value_counts()
print("Частота причин у верхних 10% по риску:")
print((counts / len(risky) * 100).round(1))

In [ ]:
# У скольких рискованных клиентов вообще нет причин для показа
n_empty = sum(1 for i in risky if len(reasons[i]) == 0)
n_short = sum(1 for i in risky if 0 < len(reasons[i]) < 3)
print(f"Без причин: {n_empty} из {len(risky)} | меньше 3 причин: {n_short}")

# Насколько топ-3 "показываемых" причин объясняют риск: доля положительного вклада, который они покрывают
pos_total = np.clip(grouped.values, 0, None).sum(axis=1)
shown = np.array([sum(v for _, v in reasons[i]) for i in range(len(reasons))])
print(f"Средняя доля положительного вклада, покрытая топ-3 причинами: "
      f"{np.mean(shown[risky] / pos_total[risky]):.1%}")

Что смотрим:
1. совпадают ли топ-признаки SHAP с тем, что мы видели по gain и корреляциям;
2. нет ли среди причин странных или непонятных формулировок;
3. не забирает ли значимую долю вклада группа, которую мы скрываем (возраст, регион);
4. не осталось ли важных признаков в группе "other".